# Stage 3-2. Layer Modules

이 노트북은 `src/nn/layers.py`에 구현된 레이어 모듈을 실습한다.

| 클래스 | 역할 |
|---|---|
| `Module` | 모든 레이어의 기반 클래스. `params`, `grads`, `training` 인터페이스 제공 |
| `Linear` | 완전연결 레이어. forward: $y = xW + b$, backward: $\nabla W$, $\nabla b$, $\nabla x$ 계산 |
| `Sigmoid` | Sigmoid 활성화 레이어. backward: $\sigma(x)(1-\sigma(x))$ |
| `ReLU` | ReLU 활성화 레이어. backward: mask 기반 gradient 전달 |
| `Sequential` | 레이어 순차 조립. forward/backward를 자동으로 체인 연결 |

**학습 목표**
1. `Linear.forward`와 `Linear.backward`의 shape 변화를 추적한다.
2. `grad_w`, `grad_b`가 `params`/`grads` 리스트에 어떻게 바인딩되는지 확인한다.
3. `Sequential`로 레이어를 조립하고 forward/backward를 한 번에 실행한다.
4. `Sigmoid`, `ReLU` backward의 gradient 흐름을 비교한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt

from src.nn.layers import Linear, Sigmoid, ReLU, Sequential, Module

## 1. Linear: forward shape 추적

In [ ]:
# Linear(784 -> 256) 생성 및 forward
layer = Linear(784, 256, seed=42)

x = np.random.randn(32, 784).astype(np.float32)  # (batch=32, in=784)
out = layer.forward(x)

print("=== Linear forward ===")
print(f"입력 x   : {x.shape}")
print(f"w        : {layer.w.shape}")
print(f"b        : {layer.b.shape}")
print(f"출력 y=xW+b: {out.shape}")

## 2. Linear: backward — grad_w, grad_b, grad_x

In [ ]:
# 상위 레이어에서 전달된 gradient (출력 shape과 동일)
dout = np.random.randn(32, 256).astype(np.float32)

grad_x = layer.backward(dout)

print("=== Linear backward ===")
print(f"dout (상위 gradient) : {dout.shape}")
print(f"grad_x (하위 전달)   : {grad_x.shape}")
print(f"grad_w               : {layer.grad_w.shape}")
print(f"grad_b               : {layer.grad_b.shape}")
print()
print("수식 검증:")
print(f"  grad_w = x.T @ dout  → ({x.shape[1]}, {dout.shape[0]}) @ ({dout.shape}) = {(x.T @ dout).shape}")
print(f"  grad_b = dout.sum(0) → {dout.sum(axis=0).shape}")
print(f"  grad_x = dout @ w.T  → ({dout.shape}) @ ({layer.w.T.shape}) = {grad_x.shape}")

> **핵심**: `grad_w`, `grad_b`는 in-place(`[...]`) 방식으로 업데이트되므로 `layer.params`/`layer.grads` 리스트가 가리키는 같은 배열이 갱신된다.

## 3. params / grads 바인딩 구조

In [ ]:
# params[0]과 layer.w는 같은 객체를 참조한다
print("layer.params[0] is layer.w  :", layer.params[0] is layer.w)
print("layer.grads[0]  is layer.grad_w :", layer.grads[0] is layer.grad_w)
print()
print("params 목록:")
for i, p in enumerate(layer.params):
    print(f"  params[{i}] shape: {p.shape}, dtype: {p.dtype}")

print()
print("grads 목록:")
for i, g in enumerate(layer.grads):
    print(f"  grads[{i}]  shape: {g.shape}, dtype: {g.dtype}")

## 4. Sigmoid 레이어 — forward / backward

In [ ]:
sig = Sigmoid()

x_s = np.array([[-2.0, -1.0, 0.0, 1.0, 2.0]], dtype=np.float32)
y_s = sig.forward(x_s)

dout_s = np.ones_like(y_s)
dx_s = sig.backward(dout_s)

print("Sigmoid forward:")
print(f"  x   : {x_s}")
print(f"  σ(x): {np.round(y_s, 4)}")
print()
print("Sigmoid backward (dout=1):")
print(f"  σ(x)·(1-σ(x)): {np.round(dx_s, 4)}")
print()
print("수식 검증:")
manual_grad = y_s * (1.0 - y_s)
print(f"  수동 계산  : {np.round(manual_grad, 4)}")
print(f"  일치 여부  : {np.allclose(dx_s, manual_grad)}")

## 5. ReLU 레이어 — forward / backward

In [ ]:
relu = ReLU()

x_r = np.array([[-2.0, -0.5, 0.0, 0.5, 2.0]], dtype=np.float32)
y_r = relu.forward(x_r)

dout_r = np.ones_like(y_r)
dx_r = relu.backward(dout_r)

print("ReLU forward:")
print(f"  x          : {x_r}")
print(f"  ReLU(x)    : {y_r}")
print()
print("ReLU backward (dout=1):")
print(f"  mask (x>0) : {relu._mask}")
print(f"  grad_x     : {dx_r}")

## 6. Sigmoid vs ReLU gradient 크기 비교

In [ ]:
x_range = np.linspace(-4, 4, 300).astype(np.float32)

# Sigmoid gradient
sig2 = Sigmoid()
_ = sig2.forward(x_range.reshape(1, -1))
sig_grad = sig2.backward(np.ones((1, len(x_range)))).flatten()

# ReLU gradient
relu2 = ReLU()
_ = relu2.forward(x_range.reshape(1, -1))
relu_grad = relu2.backward(np.ones((1, len(x_range)))).flatten()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Activation Gradients", fontsize=13, fontweight="bold")

ax1.plot(x_range, sig_grad, color="steelblue", linewidth=2)
ax1.set_title("Sigmoid: σ(x)·(1−σ(x))")
ax1.set_xlabel("x")
ax1.set_ylabel("gradient")
ax1.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax1.text(0.5, 0.9, f"max gradient: {sig_grad.max():.4f}",
         transform=ax1.transAxes, ha="center", fontsize=9, color="gray")
ax1.grid(True, alpha=0.3)

ax2.plot(x_range, relu_grad, color="mediumseagreen", linewidth=2)
ax2.set_title("ReLU: 1(x>0)")
ax2.set_xlabel("x")
ax2.set_ylabel("gradient")
ax2.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax2.text(0.5, 0.9, "gradient: 0 or 1",
         transform=ax2.transAxes, ha="center", fontsize=9, color="gray")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> **vanishing gradient**: Sigmoid의 최대 gradient는 0.25로 레이어가 깊어질수록 gradient가 소실된다.  
> ReLU는 gradient가 0 또는 1로 유지되어 깊은 네트워크에서 더 안정적으로 학습된다.

## 7. Sequential — 레이어 조립 및 forward/backward

In [ ]:
# Linear(4 -> 8) -> Sigmoid -> Linear(8 -> 3) 구성
net = Sequential(
    Linear(4, 8, seed=0),
    Sigmoid(),
    Linear(8, 3, seed=1),
)

x_net = np.random.randn(5, 4).astype(np.float32)  # (batch=5, in=4)
logits = net.forward(x_net)

print("=== Sequential forward ===")
print(f"입력  : {x_net.shape}")
print(f"출력  : {logits.shape}")
print()
print("params 목록 (총 len):", len(net.params))
for i, p in enumerate(net.params):
    print(f"  params[{i}] : {p.shape}")

In [ ]:
# backward 실행 — dout shape은 출력과 동일
dout_net = np.random.randn(5, 3).astype(np.float32)
grad_x = net.backward(dout_net)

print("=== Sequential backward ===")
print(f"dout  : {dout_net.shape}")
print(f"grad_x: {grad_x.shape}")
print()
print("grads 목록 (backward 후):")
for i, g in enumerate(net.grads):
    print(f"  grads[{i}] : {g.shape}  |  norm = {np.linalg.norm(g):.4f}")

## 8. shape 추적 표 — 784 → 256 → 128 → 10 MLP 구조

In [ ]:
# MLP 구조와 동일한 Sequential 구성 (출력 10: multiclass)
mlp_like = Sequential(
    Linear(784, 256, seed=0),
    Sigmoid(),
    Linear(256, 128, seed=1),
    Sigmoid(),
    Linear(128, 10, seed=2),
)

# 각 레이어의 입출력 shape 추적
x_trace = np.random.randn(32, 784).astype(np.float32)
current = x_trace

print(f"{'레이어':<25} {'입력 shape':<20} {'출력 shape':<20} {'params 수':>10}")
print("-" * 80)

layer_names = ["Linear(784→256)", "Sigmoid", "Linear(256→128)", "Sigmoid", "Linear(128→10)"]
for name, layer in zip(layer_names, mlp_like.layers):
    in_shape = current.shape
    current = layer.forward(current)
    out_shape = current.shape
    n_params = sum(p.size for p in layer.params)
    print(f"{name:<25} {str(in_shape):<20} {str(out_shape):<20} {n_params:>10,}")

print("-" * 80)
total = sum(p.size for p in mlp_like.params)
print(f"{'전체 파라미터 수':<45} {total:>10,}")

## 9. 정리

| 클래스 | forward | backward | params/grads |
|---|---|---|---|
| `Linear` | $y = xW + b$ | $\nabla W, \nabla b, \nabla x$ | `[w, b]` / `[grad_w, grad_b]` |
| `Sigmoid` | $\sigma(x)$ | $\sigma(x)(1-\sigma(x)) \cdot dout$ | `[]` / `[]` |
| `ReLU` | $\max(0, x)$ | mask$(x>0) \cdot dout$ | `[]` / `[]` |
| `Sequential` | 순서대로 forward | 역순으로 backward | 모든 하위 레이어 통합 |

**핵심 설계 원칙**
- `grad_w`, `grad_b`는 in-place(`[...]`)로 업데이트되어 `params`/`grads` 리스트와 메모리를 공유한다.
- optimizer는 `model.params`와 `model.grads`만 참조하면 레이어 내부 구조를 알 필요가 없다.
- `Sequential.params`/`grads`는 생성 시 한 번만 수집되므로 레이어 추가·제거 시 재구성이 필요하다.